In [12]:
import os
import json
import base64
import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# ========== 环境参数（优先从 .env 读取，否则使用默认值） ==========
HOST_IP = os.getenv("HOST_IP", "192.168.1.13")

LLM_PORT = int(os.getenv("LLM_PORT", "10001"))
VLM_PORT = int(os.getenv("VLM_PORT", "10002"))
ASR_PORT = int(os.getenv("ASR_PORT", "10003"))
EMBEDDING_PORT = int(os.getenv("EMBEDDING_PORT", "10004"))
RERANKER_PORT = int(os.getenv("RERANKER_PORT", "10005"))

API_KEY = os.getenv("API_KEY", "xtsk")


def auth_headers() -> dict:
    """构造带 Bearer Token 的请求头。"""
    return {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }


def post_json(url: str, payload: dict, timeout: int = 300) -> dict:
    """发送 POST 请求并返回 JSON 响应（非流式）。"""
    resp = requests.post(url, headers=auth_headers(), json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def iter_sse_lines(response):
    """从流式 HTTP 响应中逐行解析 SSE data 字段。"""
    for raw in response.iter_lines(decode_unicode=True):
        if not raw:
            continue
        if raw.startswith("data: "):
            yield raw[len("data: "):].strip()

In [ ]:
# ========== LLM 文本对话（requests） ==========

url = f"http://{HOST_IP}:{LLM_PORT}/v1/chat/completions"

payload = {
    "model": "LLM",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "解释什么是function call"},
    ],
    "temperature": 0.2,
}

# --- 非流式 ---
print("=== LLM 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== LLM 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        # 流式返回的增量内容在 delta.content 中
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

In [ ]:
# ========== VLM 图文理解（requests） ==========

local_image = "入院_1769051567.jpeg"  # 请确保文件存在
if not os.path.exists(local_image):
    raise FileNotFoundError(f"image file not found: {local_image}")

# 读取图片并 base64 编码
with open(local_image, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

url = f"http://{HOST_IP}:{VLM_PORT}/v1/chat/completions"

payload = {
    "model": "VLM",
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "请描述图片里最主要的内容。"},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
                },
            ],
        }
    ],
    "temperature": 0.2,
}

# --- 非流式 ---
print("=== VLM 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== VLM 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        # 流式返回的增量内容在 delta.content 中
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

In [5]:
# ========== ASR 语音识别（requests） ==========

local_audio = "di-test-sr-1.wav"  # 请确保文件存在
if not os.path.exists(local_audio):
    raise FileNotFoundError(f"audio file not found: {local_audio}")

# 读取音频并 base64 编码
with open(local_audio, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")

url = f"http://{HOST_IP}:{ASR_PORT}/v1/chat/completions"

payload = {
    "model": "ASR",
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "audio_url",
                    "audio_url": {"url": f"data:audio/wav;base64,{audio_b64}"},
                }
            ],
        }
    ],
}

# --- 非流式 ---
print("=== ASR 非流式 ===")
data = post_json(url, payload)
print(data["choices"][0]["message"]["content"])

# --- 流式 ---
print("\n=== ASR 流式 ===")
payload_stream = {**payload, "stream": True}
with requests.post(url, headers=auth_headers(), json=payload_stream,
                   stream=True, timeout=300) as response:
    response.raise_for_status()
    for sse in iter_sse_lines(response):
        if sse == "[DONE]":
            break
        event_data = json.loads(sse)
        content = event_data.get("choices", [{}])[0].get("delta", {}).get("content")
        if content:
            print(content, end="", flush=True)
print()

=== ASR 非流式 ===
language Chinese<asr_text>选择有价值的应用场景，然后研究如何通过模型能力实现功能。

=== ASR 流式 ===
language Chinese<asr_text>选择有价值的应用场景，然后研究如何通过模型能力实现功能。


In [19]:
# ========== Embedding 向量化（requests） ==========

url = f"http://{HOST_IP}:{EMBEDDING_PORT}/v1/embeddings"

payload = {
    "model": "EMBEDDING",
    "input": [
        "向量数据库适合存什么？",
        "embedding 有什么用？",
    ],
}

data = post_json(url, payload)
# 打印第一条文本的向量维度和前 20 个分量
print("dim =", len(data["data"][0]["embedding"]))
print("first 20 =", data["data"][0]["embedding"][:20])

dim = 2560
first 20 = [-0.00036868377355858684, -0.011207986623048782, 0.00996920932084322, 0.021944057196378708, -0.0016811980167403817, 0.04341620206832886, 0.0528545044362545, -0.004807636141777039, 0.00814053788781166, 0.007226201705634594, 0.007285191211849451, 0.015219265595078468, 0.0002147582999896258, 0.00819952692836523, -0.025601401925086975, 0.03161831945180893, 0.03327002376317978, 0.01628107577562332, -0.005898940376937389, -0.004188247490674257]


In [22]:
# ========== Reranker 重排序（requests） ==========
# 注意：reranker 服务使用 /rerank 端点，不走 /v1/chat/completions

url = f"http://{HOST_IP}:{RERANKER_PORT}/rerank"

payload = {
    "model": "RERANKER",
    "query": "我是一只猫？",
    "documents": [
        "我是一只狗",
        "我也是一只猫",
        "你可以在 APP 里提交申请。",
        "退税需要满足一定条件，并准备相关材料。",
        "今天天气不错。",
    ],
    "top_n": 5,
}

try:
    data = post_json(url, payload)
    print(data)
except Exception as e:
    print("reranker 调用失败:", e)

{'id': 'rerank-8d638bd4f5aaca37', 'model': 'RERANKER', 'usage': {'prompt_tokens': 55, 'total_tokens': 55}, 'results': [{'index': 1, 'document': {'text': '我也是一只猫', 'multi_modal': None}, 'relevance_score': 0.9535250663757324}, {'index': 0, 'document': {'text': '我是一只狗', 'multi_modal': None}, 'relevance_score': 0.9295282959938049}, {'index': 3, 'document': {'text': '退税需要满足一定条件，并准备相关材料。', 'multi_modal': None}, 'relevance_score': 0.6902287006378174}, {'index': 4, 'document': {'text': '今天天气不错。', 'multi_modal': None}, 'relevance_score': 0.6227165460586548}, {'index': 2, 'document': {'text': '你可以在 APP 里提交申请。', 'multi_modal': None}, 'relevance_score': 0.5640547871589661}]}
